# Build the feature CSV → save it to Google Drive

Clones **[GazeVLM-HWSW-Codesign](https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign)**,
runs the repo's own script on **2 videos**, and saves the result to Drive so it outlives
the Colab session.

```bash
python -m src.dataprep.build_feature_csv --n_videos 2 --cleanup_raw
```

## Columns

| Column | Per row | What |
|---|---|---|
| `idx`, `sequence` | 1 | global row index, video name |
| `feat_frame_1`, `feat_frame_2` | 1 | paths to the frozen DINOv2 `.npz` |
| `frame_similarity` | 1 | CLS cosine — teacher label |
| `gaze_patch_token_sim` | 1 | gaze-patch cosine — teacher label |
| `n_velocities` | 1 | steps in the window (**9**) |
| `gaze_rates_window` | **9 × 3** | ω_yaw, ω_pitch, ω_mag per step |
| `n_gaze_in_gap` | 1 | raw gaze samples in the window (**10**) |
| `n_oof_in_gap` | 1 | how many of them fell outside the lens |
| `gaze_xy_window` | **10 × 2** | projected image coords, normalised |
| `gaze_vec3d_rates_window` | **9 × 3** | d/dt of the 3D unit gaze vector (1/s) |
| `n_imu_in_gap` | 1 | **raw** IMU samples in the window (~1000) |
| `imu_window` | **10 × 6** | accel xyz (m/s²) + gyro xyz (rad/s), binned |
| `imu_accel_mag_mean` | 1 | mean ‖accel‖ over the raw samples |
| `imu_gyro_mag_mean` | 1 | mean ‖gyro‖ over the raw samples |

The last six are new. Note the shapes differ by one: **10 samples give 9 velocities**, so
both rate columns are the differences *between* the points in `gaze_xy_window`.

**`gaze_vec3d_rates_window` is the better speed signal.** It is the 3D counterpart of
`gaze_rates_window`, same 9 × 3 shape, so they line up step for step — but because the
gaze vector lies on the unit sphere, the **norm of its three channels is the exact
angular speed**. `ω_mag` only approximates that: it treats yaw and pitch as orthogonal
and overstates the yaw term by 1/cos(pitch), which is ~21% at the −34° pitch this data
sits at. Measured on a synthetic sweep: the norm matches the exact arccos speed to
**0.003%**, `ω_mag` to within **8.6%**. Its three channels carry only two degrees of
freedom, though — d**v**/dt is perpendicular to **v**.

**`imu_window` is binned, not raw.** Aria's IMU runs at ~1 kHz, so a 1 s window holds
~1000 × 6 values — about 50 KB per CSV row, over a GB across all 143 sequences. The
default `--imu_hz 10` averages into 10 bins so the IMU column lines up one-to-one with
`gaze_xy_window`. Pass `--imu_hz 0` for every raw sample, and check the file size before
scaling up. The two magnitude columns are computed on the **unbinned** samples, because
averaging vectors within a bin cancels opposing motion and understates how much the head
really moved.

## The CSV alone is not enough

`feat_frame_1` and `feat_frame_2` are **paths** to `.npz` feature files. Saving only the
CSV would leave those paths pointing at `/content/...`, which disappears when the runtime
ends — the file would look fine and be useless.

So this notebook copies **both**, and rewrites the paths inside the CSV to their Drive
locations. The result is self-contained: reload it in any future session and it works.

| Saved to Drive | What it is |
|---|---|
| `feature_dataset.csv` | the table, with Drive paths |
| `features/<sequence>/feat_*.npz` | the frozen DINOv2 features |

## Cost

2 videos ≈ **5 GB** downloaded (deleted afterwards by `--cleanup_raw`), ~13 MB kept.
Expect ~4 min. **No GPU needed.**

## 1 — Clone the repo and install

In [ ]:
!git clone -q https://github.com/shubhamOjha1000/GazeVLM-HWSW-Codesign.git /content/GazeVLM
%cd /content/GazeVLM
!git log --oneline -1

!pip -q install -r requirements.txt
!pip -q install projectaria-tools

import os, glob, json, shutil, time
import numpy as np, pandas as pd
print("\nrepo:", os.getcwd())

## 2 — Mount Drive and pick where things go

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/GazeVLM"        # <- change if you like
DRIVE_FEATS = os.path.join(DRIVE_DIR, "features")
DRIVE_CSV   = os.path.join(DRIVE_DIR, "feature_dataset.csv")
os.makedirs(DRIVE_FEATS, exist_ok=True)

print("will save to:")
print("   ", DRIVE_CSV)
print("   ", DRIVE_FEATS + "/<sequence>/feat_*.npz")

## 3 — The download-links JSON

If you already keep it in Drive, use Option B and skip the upload every session.

In [ ]:
# ---- Option A: upload from your laptop ----
from google.colab import files
up = files.upload()                       # pick your aea_download_urls.json
URLS_JSON = "/content/" + list(up.keys())[0]
os.rename(list(up.keys())[0], URLS_JSON)

# ---- Option B: already in Drive ----
# URLS_JSON = os.path.join(DRIVE_DIR, "aea_download_urls.json")

# keep a copy in Drive so future sessions can use Option B
kept = os.path.join(DRIVE_DIR, "aea_download_urls.json")
if os.path.abspath(URLS_JSON) != os.path.abspath(kept):
    shutil.copy2(URLS_JSON, kept)
    print("copied the JSON to Drive for next time:", kept)

print(f"{len(json.load(open(URLS_JSON))['sequences'])} videos available")
print("NOTE: the links expire after ~14 days -- re-download the JSON when builds fail.")

## 4 — Build, on local disk

Built in `/content` rather than straight to Drive: writing hundreds of small `.npz` files
over the Drive mount is far slower. They get copied across in one go afterwards.

Every video is used in full (~190 rows each at 1 FPS), so expect **~380 rows**.

The per-sequence table printed at the end reports `samples_per_row` (≈ 10.0 expected),
`oof_samples` (should be small) and `imu_per_row` (≈ 1000 — the **raw** count feeding
each 10-bin `imu_window`).

IMU extraction adds ~30–60 s per video. It reads the VRS already on disk for the
calibration, so it costs no extra download. Pass `--no_imu` to skip it.

In [ ]:
N_VIDEOS  = 2
SEED      = 0
IMU_HZ    = 10       # bins/s for imu_window; 10 lines up with the ~10 gaze samples.
                     # 0 = every raw sample (~50 KB per row -- watch the file size).
LOCAL_CSV = "/content/data/feature_dataset.csv"
LOCAL_FEAT = "/content/data/features"

t0 = time.time()
!python -m src.dataprep.build_feature_csv \
    --urls_json  "{URLS_JSON}" \
    --out_csv    "{LOCAL_CSV}" \
    --raw_dir    /content/data/raw \
    --frames_dir /content/data/frames_1fps \
    --feat_dir   "{LOCAL_FEAT}" \
    --n_videos   {N_VIDEOS} \
    --seed       {SEED} \
    --imu_hz     {IMU_HZ} \
    --cleanup_raw

print(f"\nbuild took {(time.time()-t0)/60:.1f} min")

## 5 — Copy to Drive and rewrite the paths

This is the step that makes the saved CSV actually reusable.

In [ ]:
# --- features ---------------------------------------------------------------
t0 = time.time()
n_files = 0
for seq in sorted(os.listdir(LOCAL_FEAT)):
    src_dir = os.path.join(LOCAL_FEAT, seq)
    dst_dir = os.path.join(DRIVE_FEATS, seq)
    if not os.path.isdir(src_dir):
        continue
    os.makedirs(dst_dir, exist_ok=True)
    for f in sorted(os.listdir(src_dir)):
        shutil.copy2(os.path.join(src_dir, f), os.path.join(dst_dir, f))
        n_files += 1
    print(f"   {seq}: {len(os.listdir(dst_dir))} files")
print(f"copied {n_files} feature files in {time.time()-t0:.0f}s")

# --- CSV, with paths pointing at Drive --------------------------------------
df = pd.read_csv(LOCAL_CSV)

def to_drive(p):
    """/content/data/features/<seq>/feat_x.npz -> <DRIVE_FEATS>/<seq>/feat_x.npz
    Keeps the <sequence>/<file> tail: the basename alone is NOT unique, since every
    sequence folder contains a feat_00000.npz."""
    parts = str(p).replace("\\", "/").rstrip("/").split("/")
    return os.path.join(DRIVE_FEATS, *parts[-2:])

for col in ("feat_frame_1", "feat_frame_2"):
    df[col] = df[col].apply(to_drive)

df.to_csv(DRIVE_CSV, index=False)
print(f"\nwrote {len(df)} rows x {df.shape[1]} columns -> {DRIVE_CSV}")
print(f"   {os.path.getsize(DRIVE_CSV)/1e3:.0f} KB")
print(f"\ncolumns: {list(df.columns)}")
print(f"\nexample rewritten path:\n   {df.loc[0, 'feat_frame_1']}")

## 6 — Verify what landed in Drive

Reads the CSV **back from Drive**, opens the feature files it points at, and recomputes a
similarity from them. If this passes, the saved dataset is self-contained.

In [ ]:
chk = pd.read_csv(DRIVE_CSV)
print(f"{len(chk)} rows x {chk.shape[1]} columns from {chk['sequence'].nunique()} videos")
display(chk.groupby("sequence").size().rename("rows").to_frame())

missing = [p for p in pd.concat([chk.feat_frame_1, chk.feat_frame_2]).unique()
           if not os.path.exists(p)]
print(f"\nreferenced feature files missing: {len(missing)}")

z0 = np.load(chk.loc[0, "feat_frame_1"])
z1 = np.load(chk.loc[0, "feat_frame_2"])
fs = float(np.dot(z0["cls"], z1["cls"]))            # tokens are L2-normalised

g = int(z0["grid"])
def cell(z):
    x, y = z["gaze_xy"]
    gx = min(g-1, int(np.clip(x, 0, 1)*g)); gy = min(g-1, int(np.clip(y, 0, 1)*g))
    return z["patches"][gy*g + gx]
ps = float(np.dot(cell(z0), cell(z1)))

ok = (len(missing) == 0
      and abs(fs - chk.loc[0, "frame_similarity"]) < 1e-3
      and abs(ps - chk.loc[0, "gaze_patch_token_sim"]) < 1e-3)
print(f"\nrecomputed from Drive vs the CSV:")
print(f"   frame_similarity     {fs:.4f}  vs  {chk.loc[0,'frame_similarity']:.4f}")
print(f"   gaze_patch_token_sim {ps:.4f}  vs  {chk.loc[0,'gaze_patch_token_sim']:.4f}")
print(f"\n   [{'PASS' if ok else 'FAIL'}]  the dataset in Drive is self-contained")

sz = sum(os.path.getsize(os.path.join(r, f))
         for r, _, fs_ in os.walk(DRIVE_FEATS) for f in fs_) / 1e6
print(f"\nDrive usage: {sz:.1f} MB of features + {os.path.getsize(DRIVE_CSV)/1e3:.0f} KB CSV")
display(chk.head(5))

## 7 — The new per-window columns

`gaze_xy_window` says **where** the eye pointed (10 samples); `gaze_rates_window` and
`gaze_vec3d_rates_window` say **how fast it moved** (9 steps, the differences between
those samples); `imu_window` says how the head moved (10 bins).

All unpack with the repo's own `parse_rates`, which splits on `;` then `,` and therefore
handles the 2-, 3- and 6-wide forms identically.

In [ ]:
import sys; sys.path.insert(0, "/content/GazeVLM")
from src.loss1.dataset import parse_rates

ROW = 0
r = chk.loc[ROW]

XY    = parse_rates(r["gaze_xy_window"])              # (n, 2)   position
RATE  = parse_rates(r["gaze_rates_window"])           # (n-1, 3) yaw/pitch rates
VRATE = parse_rates(r["gaze_vec3d_rates_window"])     # (n-1, 3) 3D vector rates
IMU   = parse_rates(r["imu_window"])                  # (n_bins, 6)

print(f"row {ROW}   sequence {r['sequence']}")
print(f"   n_gaze_in_gap {r['n_gaze_in_gap']}   n_oof_in_gap {r['n_oof_in_gap']}   "
      f"n_velocities {r['n_velocities']}   n_imu_in_gap {r['n_imu_in_gap']}\n")
print(f"   gaze_xy_window           -> {XY.shape}    where the eye pointed")
print(f"   gaze_rates_window        -> {RATE.shape}    yaw/pitch rates + magnitude")
print(f"   gaze_vec3d_rates_window  -> {VRATE.shape}    d/dt of the unit vector")
print(f"   imu_window               -> {IMU.shape}   BINNED from {r['n_imu_in_gap']} raw\n")

print("   step |      x        y   |    dvx       dvy       dvz   |  ||dv/dt||  omega_mag")
print("   -----+-------------------+------------------------------+---------------------")
for k in range(len(VRATE)):
    nrm = float(np.linalg.norm(VRATE[k]))
    print(f"   {k:4d} | {XY[k,0]:8.4f} {XY[k,1]:7.4f} | "
          f"{VRATE[k,0]:+9.4f} {VRATE[k,1]:+9.4f} {VRATE[k,2]:+9.4f} | "
          f"{nrm:9.4f} {RATE[k,2]:10.4f}")
print(f"   {len(VRATE):4d} | {XY[-1,0]:8.4f} {XY[-1,1]:7.4f} |  (one more position than steps)")

print("\n   The last two columns are both 'gaze speed'. ||dv/dt|| is exact; omega_mag")
print("   treats yaw and pitch as orthogonal and reads HIGH by 1/cos(pitch).")

print("\n   step |   accel x       y       z   |    gyro x       y       z")
print("   -----+-----------------------------+-----------------------------")
for k in range(len(IMU)):
    print(f"   {k:4d} | {IMU[k,0]:+8.3f} {IMU[k,1]:+8.3f} {IMU[k,2]:+8.3f} | "
          f"{IMU[k,3]:+8.4f} {IMU[k,4]:+8.4f} {IMU[k,5]:+8.4f}")
print(f"\n   accel z should sit near +/-9.81 -- that is gravity, not motion.")
print(f"   imu_accel_mag_mean {r['imu_accel_mag_mean']:.4f} m/s^2   "
      f"imu_gyro_mag_mean {r['imu_gyro_mag_mean']:.4f} rad/s")

### Checks

Eight things worth asserting before this data is trusted downstream.

In [ ]:
XYs    = [parse_rates(s) for s in chk["gaze_xy_window"]]
VRATEs = [parse_rates(s) for s in chk["gaze_vec3d_rates_window"]]
RATEs  = [parse_rates(s) for s in chk["gaze_rates_window"]]
IMUs   = [parse_rates(s) for s in chk["imu_window"]]

n_xy  = np.array([len(a) for a in XYs])
n_vr  = np.array([len(a) for a in VRATEs])
n_rat = chk["n_velocities"].to_numpy()

allxy  = np.concatenate([a for a in XYs if len(a)])
allvr  = np.concatenate([a for a in VRATEs if len(a)])
allrat = np.concatenate([a for a in RATEs if len(a)])
allimu = np.concatenate([a for a in IMUs if len(a)]) if any(len(a) for a in IMUs) \
         else np.zeros((0, 6))

speed_vec = np.linalg.norm(allvr, axis=1)     # exact angular speed
speed_mag = allrat[:, 2]                      # omega_mag, the approximation

checks = [
    ("vec3d rates are one SHORTER than xy",
     bool((n_vr == n_xy - 1).all()),
     f"{int(np.median(n_vr))} steps from {int(np.median(n_xy))} samples"),

    ("length matches n_gaze_in_gap",
     bool((n_xy == chk['n_gaze_in_gap'].to_numpy()).all()),
     f"median {int(np.median(n_xy))} samples per row"),

    ("both rate columns agree in length",
     bool((n_vr == n_rat).all()),
     f"{int((n_vr != n_rat).sum())} rows disagree"),

    ("omega_mag >= ||dv/dt||  (it over-reads by 1/cos(pitch))",
     bool((speed_mag >= speed_vec - 1e-4).mean() > 0.99),
     f"omega_mag is {100*np.nanmedian(speed_mag/np.maximum(speed_vec,1e-9)-1):+.1f}% "
     f"high at the median"),

    ("xy inside the image",
     bool((allxy >= 0).all() and (allxy <= 1).all()),
     f"x [{allxy[:,0].min():.3f}, {allxy[:,0].max():.3f}]  "
     f"y [{allxy[:,1].min():.3f}, {allxy[:,1].max():.3f}]"),

    ("IMU present on every row",
     bool((chk['n_imu_in_gap'].to_numpy() > 0).all()),
     f"{int((chk['n_imu_in_gap'] == 0).sum())} rows have no IMU"),

    ("no NaN in imu_window (every bin caught a sample)",
     bool(not np.isnan(allimu).any()),
     f"{int(np.isnan(allimu).any(axis=1).sum())} empty bins out of {len(allimu)}"),

    ("accel magnitude ~ 1 g (gravity dominates)",
     bool(5.0 < float(chk['imu_accel_mag_mean'].median()) < 15.0),
     f"median {chk['imu_accel_mag_mean'].median():.2f} m/s^2  (g = 9.81)"),
]

print(f"{len(chk)} rows, {len(allxy):,} gaze samples total\n")
for name, okk, detail in checks:
    print(f"   [{'PASS' if okk else 'FAIL'}]  {name:36s} {detail}")

oof = int(chk["n_oof_in_gap"].sum())
print(f"\n   out-of-FOV samples: {oof:,} / {len(allxy):,} ({100*oof/len(allxy):.2f}%)")
print("   Those carry an INVENTED (0.5, 0.5) in gaze_xy_window and are indistinguishable")
print("   from a real centre gaze -- filter on n_oof_in_gap before using the xy column.")
print("   Both rate columns are unaffected: they need no camera model.")

### The two speed signals, side by side

The third panel is the point of the new column: every dot below the diagonal is a step
where `ω_mag` claims the eye moved faster than it did.

In [ ]:
import matplotlib.pyplot as plt

SEQ = chk["sequence"].iloc[0]
sub = chk[chk["sequence"] == SEQ]
XYs_s = np.concatenate([parse_rates(s) for s in sub["gaze_xy_window"]])
VRs_s = np.concatenate([parse_rates(s) for s in sub["gaze_vec3d_rates_window"]])

fig, ax = plt.subplots(1, 3, figsize=(16, 4.6))

ax[0].scatter(XYs_s[:, 0], XYs_s[:, 1], s=3, alpha=.25)
ax[0].set_xlim(0, 1); ax[0].set_ylim(1, 0)      # image rows grow downward
ax[0].set_xlabel("x"); ax[0].set_ylabel("y")
ax[0].set_title(f"gaze_xy_window — {len(XYs_s):,} samples")

ax[1].plot(VRs_s[:200, 0], lw=.8, label="dvx/dt")
ax[1].plot(VRs_s[:200, 1], lw=.8, label="dvy/dt")
ax[1].plot(VRs_s[:200, 2], lw=.8, label="dvz/dt")
ax[1].axhline(0, lw=.4, c="k"); ax[1].legend(fontsize=8)
ax[1].set_xlabel("step"); ax[1].set_ylabel("1/s")
ax[1].set_title("gaze_vec3d_rates_window — first 200 steps")

lim = float(max(speed_vec.max(), speed_mag.max())) * 1.05
ax[2].scatter(speed_vec, speed_mag, s=4, alpha=.2)
ax[2].plot([0, lim], [0, lim], "r--", lw=1, label="y = x")
ax[2].set_xlim(0, lim); ax[2].set_ylim(0, lim); ax[2].legend()
ax[2].set_xlabel("||dv/dt||   (exact)"); ax[2].set_ylabel("omega_mag   (approximation)")
ax[2].set_title("omega_mag reads HIGH — that is the 1/cos(pitch) error")

plt.tight_layout(); plt.show()

print("Left  : should cluster where the wearer looked, NOT hug the borders.")
print("Middle: signed channels crossing zero as the eye reverses; spikes are saccades.")
print("Right : points sit ABOVE y=x. The gap is how much omega_mag overstates the speed;")
print(f"        median inflation here is {100*np.nanmedian(speed_mag/np.maximum(speed_vec,1e-9)-1):+.1f}%.")

## 8 — What the IMU is actually for: testing the premise, free

The whole project rests on a two-link chain:

```
eye velocity  ──(VOR)──►  head motion  ──►  the picture changes
```

Until now only the **ends** were measurable — gaze rates and `frame_similarity` — so a
weak correlation could not be blamed on either link. `imu_gyro_mag_mean` measures the
**middle** directly, which splits the chain in two and says *which link is weak*.

This costs no training and no extra download. It is the cheapest real result in the
project, and it is worth reading before spending hours on a 143-video build.

In [ ]:
def _rank(v):
    o = np.argsort(np.argsort(v)); return o.astype(float)

def r_of(a, b, rank=False):
    """Pearson, or Spearman when rank=True.

    Spearman matters here: the rate data is dominated by saccade spikes an order of
    magnitude above the median, and Pearson is driven by those few points. A rank
    correlation asks the question we actually care about -- do faster windows have less
    similar frames -- without letting six outliers decide the answer.
    """
    a, b = np.asarray(a, float), np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 3:
        return float("nan")
    x, y = (a[m], b[m]) if not rank else (_rank(a[m]), _rank(b[m]))
    return float(np.corrcoef(x, y)[0, 1])

# the EXACT angular speed: norm of d(unit vector)/dt, not omega_mag
gaze_mag = np.array([np.linalg.norm(parse_rates(s), axis=1).mean() if s else np.nan
                     for s in chk["gaze_vec3d_rates_window"]])
head_mag = chk["imu_gyro_mag_mean"].to_numpy()
frame_s  = chk["frame_similarity"].to_numpy()
gaze_s   = chk["gaze_patch_token_sim"].to_numpy()

link1, link2, whole = (r_of(gaze_mag, head_mag), r_of(head_mag, frame_s),
                       r_of(gaze_mag, frame_s))
s1, s2, sw = (r_of(gaze_mag, head_mag, True), r_of(head_mag, frame_s, True),
              r_of(gaze_mag, frame_s, True))

print("=" * 78)
print("                                              Pearson   Spearman   want")
print(f"  LINK 1   gaze speed  vs head speed (VOR)    {link1:+.3f}    {s1:+.3f}    POSITIVE")
print(f"  LINK 2   head speed  vs frame_similarity    {link2:+.3f}    {s2:+.3f}    NEGATIVE")
print("  " + "-" * 62)
print(f"  WHOLE    gaze speed  vs frame_similarity    {whole:+.3f}    {sw:+.3f}    NEGATIVE")
print(f"           gaze speed  vs gaze_patch_token    {r_of(gaze_mag, gaze_s):+.3f}    "
      f"{r_of(gaze_mag, gaze_s, True):+.3f}    NEGATIVE")
print("=" * 78)

# The verdict checks SIGN, not just magnitude. A wrong-signed correlation is not a
# weak result, it is evidence something is wired backwards -- and |r| alone cannot
# tell the two apart.
if link2 > 0.1:
    print("\n  LINK 2 HAS THE WRONG SIGN. More head motion is associated with MORE")
    print("  similar frames, which is backwards. Suspect the frame<->IMU time")
    print("  alignment (all three streams are rebased to their own first sample)")
    print("  before reading anything into the premise.")
elif link2 > -0.1:
    print("\n  LINK 2 IS FLAT. Head motion does not predict frame change here, so no")
    print("  gaze signal could either -- gaze reaches the picture only THROUGH head")
    print("  motion. Check alignment and scene content before blaming the premise.")
elif link1 < -0.1:
    print("\n  LINK 1 HAS THE WRONG SIGN. Faster eyes with slower head is the opposite")
    print("  of VOR. Check that gaze is in CPF (head) coordinates, not world-stabilised.")
elif link1 < 0.1:
    print("\n  LINK 1 IS FLAT. Head motion does move the picture, but eye velocity is")
    print("  not tracking head velocity -- the VOR assumption is what fails, and it is")
    print("  the load-bearing one. IMU would work as a gate input; gaze alone would not.")
elif whole > -0.1:
    print("\n  BOTH LINKS HOLD BUT THE CHAIN DOES NOT. Each half works, the end-to-end")
    print("  claim does not. That is the worst case for this project, because it is")
    print("  exactly the path the gate depends on.")
else:
    print("\n  THE CHAIN HOLDS END TO END on this sample. Worth scaling up.")

# If head motion is the ONLY path from eye to pixels, the chain is Markov and
# r_whole ~= r_link1 * r_link2. A |whole| well ABOVE that product means gaze carries
# information about the picture that does NOT go through head motion at all.
pred = link1 * link2
print(f"\n  Markov check:  link1 x link2 = {pred:+.3f}   vs measured whole = {whole:+.3f}")
if np.isfinite(pred) and np.isfinite(whole):
    if whole < pred - 0.08:
        print("  |whole| exceeds the product -- gaze predicts the picture through some")
        print("  route BESIDES head motion (saccades relocate the attended region without")
        print("  moving the head). Good news for the gate: more signal than VOR alone.")
    elif whole > pred + 0.08:
        print("  |whole| falls short of the product -- the two links do not compose,")
        print("  so the linear chain model is too simple here (or noise dominates).")
    else:
        print("  Consistent with a pure chain: gaze -> head -> pixels, nothing else.")

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))
for a_, (x, y, xl, yl, ttl) in zip(ax, [
        (gaze_mag, head_mag, "gaze speed (rad/s)", "head speed (rad/s)",
         f"LINK 1 — VOR:  r = {link1:+.3f}"),
        (head_mag, frame_s, "head speed (rad/s)", "frame_similarity",
         f"LINK 2 — motion to pixels:  r = {link2:+.3f}"),
        (gaze_mag, frame_s, "gaze speed (rad/s)", "frame_similarity",
         f"END TO END:  r = {whole:+.3f}")]):
    a_.scatter(x, y, s=14, alpha=.45, edgecolor="k", linewidth=.2)
    a_.set_xlabel(xl); a_.set_ylabel(yl); a_.set_title(ttl)
plt.tight_layout(); plt.show()

nv = chk["sequence"].nunique()
print(f"\nHOW MUCH TO TRUST THIS: {len(chk)} rows, but only {nv} videos.")
print(f"   Naive standard error on r is 1/sqrt(n-3) = {1/np.sqrt(len(chk)-3):.3f}, so |r| >")
print(f"   {2/np.sqrt(len(chk)-3):.2f} would look 'significant' -- but consecutive rows")
print("   within a video are strongly autocorrelated, so the EFFECTIVE sample size is")
print("   nearer the number of videos than the number of rows. With a handful of videos")
print("   the honest interval is far wider than that number suggests.")
print("   Treat a near-zero result as INCONCLUSIVE, not negative. Rerun at 10+ videos.")

if nv > 1:
    print("\nPER VIDEO (does the sign survive within each scene?):")
    for s, g in chk.groupby("sequence"):
        gm = np.array([np.linalg.norm(parse_rates(x), axis=1).mean() if x else np.nan
                       for x in g["gaze_vec3d_rates_window"]])
        print(f"   {s[:34]:34s} n={len(g):4d}  link2 {r_of(g['imu_gyro_mag_mean'], g['frame_similarity']):+.3f}"
              f"   whole {r_of(gm, g['frame_similarity']):+.3f}")
    print("   A correlation that flips sign between videos is a property of the scenes,")
    print("   not of the premise.")

---

## Using it in a later session

```python
from google.colab import drive; drive.mount("/content/drive")
CSV = "/content/drive/MyDrive/GazeVLM/feature_dataset.csv"
```

```bash
python -m src.loss1.train --csv $CSV --out_dir runs/loss1
python -m src.loss2.train --csv $CSV --out_dir runs/loss2
```

No rebuild, no re-download, no DINOv2 — the paths already point into Drive.

Training reads columns **by name**, so the new ones are simply ignored by `Loss1Dataset`
and `Loss2Dataset` — both still read `gaze_rates_window`. Nothing downstream needs
changing to consume this CSV.

To train on the **exact** speed signal instead, point the dataset at the new column:

```python
Loss1Dataset(csv, ..., rates_col="gaze_vec3d_rates_window")
```

It is the same 9 × 3 shape, so `GazeRateEncoder` takes it unchanged — only the channel
statistics differ, and those are computed from the split at load time.

If you ever move the `features/` folder, pass `--feat_root <new location>` instead of
rewriting the CSV; the loaders re-root on the `<sequence>/<file>` tail.

## Adding more videos later

Change `SEED` (or pass `--seqs` with explicit names), rerun, and **write to a different
`--out_csv`**, then concatenate:

```python
big = pd.concat([pd.read_csv(a), pd.read_csv(b)], ignore_index=True)
big["idx"] = range(len(big))          # idx must stay unique and contiguous
big.to_csv("/content/drive/MyDrive/GazeVLM/feature_dataset.csv", index=False)
```

Feature folders are named per sequence, so they merge without collisions.

**An older CSV cannot be merged with a new one** — it has 8 columns, this has 16, and
`pd.concat` would fill the missing eight with `NaN` rather than erroring. Rebuild
instead: none of the new columns can be backfilled without the VRS (~2.5 GB per video),
since both the gaze projection and the IMU stream live in it.

## The IMU is not part of the gate

It is here to **diagnose**, not to deploy. The premise is that the gate runs on gaze
alone — that is the entire point, since eye tracking is already on and reading the IMU at
1 kHz is not free. `imu_window` exists so you can (a) test the two links separately in
section 8 and (b) train an IMU-input model as an **upper bound**: if IMU predicts
`frame_similarity` well and gaze does not, the ceiling is the VOR assumption, not the
architecture.

If you ever do want IMU at inference, `src/inference/gate.py` would need a second input
branch — it currently takes `(L, 3)` gaze rates only.

## A note on scale

2 videos gives ~380 rows and a 1 train / 1 val split — enough to check the plumbing, not
enough to train anything meaningful. The models carry ~1.5 M parameters. Build up to tens
of videos in Drive before drawing conclusions from any training run.